In [1]:
!pip install -q kubernetes

In [7]:
# 1. Instalación de la librería de Kubernetes
!pip install -q kubernetes

from kubernetes import client, config
import boto3
import re
from botocore.client import Config

# --- CONFIGURACIÓN DE NOMBRES ---
NAMESPACE = "sentinel-t-moviles"
ISVC_NAME = "sentinel"
BUCKET_NAME = "modelos-prod"
DATA_CONNECTION_NAME = "minio-modelos-prod" # El nombre que le pusiste en la UI de RHOAI

# --- 2. LÓGICA PARA DETECTAR LA ÚLTIMA VERSIÓN EN MINIO ---
def get_latest_v():
    s3 = boto3.client('s3', 
                      endpoint_url="http://minio-service.t-moviles-ai.svc.cluster.local:9000",
                      aws_access_key_id="minio", 
                      aws_secret_access_key="minio123",
                      config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
                      region_name='us-east-1')
    
    res = s3.list_objects_v2(Bucket=BUCKET_NAME, Delimiter='/')
    prefixes = res.get('CommonPrefixes', [])
    
    versions = []
    for p in prefixes:
        match = re.search(r'v(\d+)', p['Prefix'])
        if match:
            versions.append(int(match.group(1)))
    
    latest_v = f"v{max(versions)}" if versions else "v1"
    return latest_v

# --- 3. EJECUCIÓN DEL PATCH EN OPENSHIFT ---
def trigger_deployment():
    nueva_version = get_latest_v()
    print(f"📦 Detectada versión más reciente en S3: {nueva_version}")

    # Cargar credenciales del clúster (In-Cluster)
    try:
        config.load_incluster_config()
    except:
        config.load_kube_config()

    custom_api = client.CustomObjectsApi()

    # Estructura del Patch específica para ModelMesh/RHOAI
    patch_body = {
        "spec": {
            "predictor": {
                "model": {
                    "modelFormat": {"name": "onnx"},
                    "storage": {
                        "key": DATA_CONNECTION_NAME,
                        "path": f"{nueva_version}/"
                    }
                }
            }
        }
    }

    try:
        print(f"🔄 Aplicando patch al InferenceService '{ISVC_NAME}' en el namespace '{NAMESPACE}'...")
        
        custom_api.patch_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=NAMESPACE,
            plural="inferenceservices",
            name=ISVC_NAME,
            body=patch_body
        )
        
        print(f"🚀 ¡Éxito! El servidor de Sentinel ahora apunta a la ruta: {nueva_version}/")
        print("💡 OpenShift iniciará un Rolling Update para cargar el nuevo modelo sin caída de servicio.")

    except Exception as e:
        print(f"❌ Error al intentar actualizar: {e}")
        if "403" in str(e):
            print("\n⚠️  TIP: Ejecutá 'oc adm policy add-role-to-user edit ...' para dar permisos al Notebook.")

# Disparar deploy
trigger_deployment()

📦 Detectada versión más reciente en S3: v2
🔄 Aplicando patch al InferenceService 'sentinel' en el namespace 'sentinel-t-moviles'...
🚀 ¡Éxito! El servidor de Sentinel ahora apunta a la ruta: v2/
💡 OpenShift iniciará un Rolling Update para cargar el nuevo modelo sin caída de servicio.
